# TabDPT — CPU

Dùng **TabDPT v1.1** để bám sát thế hệ mô hình trong paper, tránh vô tình chạy v1.3/Turbo mặc định. Repository: https://github.com/layer6ai-labs/TabDPT-inference . Một số bản có các tham số dự đoán khác nhau; notebook phát hiện chữ ký hàm và không thay đổi mô hình.

**Lưu ý:** Chạy trên CPU, không pretrain từ đầu. Lần chạy đầu cần Internet để cài package/tải checkpoint. Nếu không có package/checkpoint, notebook ghi trạng thái lỗi; **không tự thay bằng mô hình giả lập**.

## 1. Cài thư viện
Chạy lệnh dưới đây trong Terminal hoặc bỏ dấu `#` ở cell tiếp theo. Nên dùng **Python 3.10–3.11** trong môi trường ảo riêng cho mỗi mô hình để tránh xung đột dependency.

In [1]:
%pip install 'tabdpt==1.1.*' scikit-learn pandas torch psutil

  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 1.7 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 3.6 MB/s  0:00:00 eta 0:00:01
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
  Attempting uninstall: huggingface-hub━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/6 [httpcore]
    Found existing installation: huggingface_hub 2.2.0━━━━━━━━ 1/6 [httpcore]
    Uninstalling huggingface_hub-2.2.0:━━━━━━━━━━━━━━━━━━━━━━━ 1/6 [httpcore]
      Successfully uninstalled huggingface_hub-2.2.0━━━━━━━━━━ 1/6 [httpcore]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [tabdpt]2m4/6 [huggingface-hub]
Note: you may need to restart the kernel to use updated packages.


## 2. Đọc dữ liệu và chia tập thống nhất

In [2]:
from pathlib import Path
import os, json, time, traceback, platform, importlib.metadata
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, log_loss

ROOT = Path.cwd().resolve()
if not (ROOT / 'datasets' / 'breast_cancer_wisconsin.csv').exists():
    if (ROOT.parent / 'datasets' / 'breast_cancer_wisconsin.csv').exists():
        ROOT = ROOT.parent
    else:
        raise FileNotFoundError('Mở notebook từ thư mục project hoặc chạy: os.chdir(duong_dan_giai_nen_ZIP)')
DATASET = ROOT / 'datasets' / 'breast_cancer_wisconsin.csv'
RESULTS = ROOT / 'results'; RESULTS.mkdir(exist_ok=True)
SPLITS = ROOT / 'splits'; SPLITS.mkdir(exist_ok=True)
MODEL_ID = 'TabDPT'
SEEDS = [42]                # Sau khi chạy thử: [42, 43, 44, 45, 46]
TRAIN_SIZES = [50, 100]    # Sau khi chạy thử: [50, 100, 200, 300]
N_CPU_THREADS = min(4, os.cpu_count() or 2)
os.environ['CUDA_VISIBLE_DEVICES'] = ''
os.environ['OMP_NUM_THREADS'] = str(N_CPU_THREADS)

frame = pd.read_csv(DATASET)
X = frame.drop(columns=['row_id', 'target']).to_numpy(dtype=np.float32)
y = frame['target'].to_numpy(dtype=np.int64)
assert len(np.unique(y)) == 2 and len(frame) == 569

def indices_for_seed(seed):
    path = SPLITS / f'breast_cancer_seed{seed}.json'
    if path.exists():
        s = json.loads(path.read_text(encoding='utf-8'))
    else:
        ids = np.arange(len(y))
        trainpool, test = train_test_split(ids, test_size=0.2, stratify=y, random_state=seed)
        train, val = train_test_split(trainpool, test_size=0.2, stratify=y[trainpool], random_state=seed)
        s = {'seed': seed, 'train': sorted(map(int,train)), 'val': sorted(map(int,val)), 'test': sorted(map(int,test))}
        path.write_text(json.dumps(s,indent=2),encoding='utf-8')
    assert not (set(s['train']) & set(s['test']))
    assert not (set(s['val']) & set(s['test']))
    assert not (set(s['train']) & set(s['val']))
    return s

def make_fold(seed, n):
    s = indices_for_seed(seed)
    pool = np.asarray(s['train'],dtype=int)
    if n > len(pool):
        raise ValueError(f'n_train={n} > train pool {len(pool)}')
    chosen, _ = train_test_split(pool, train_size=n, stratify=y[pool], random_state=seed + 101)
    chosen = np.sort(chosen)
    return (X[chosen], y[chosen], X[s['val']], y[s['val']], X[s['test']], y[s['test']], chosen, s)

def metrics(y_true, y_pred, prob):
    p = np.asarray(prob,dtype=float)
    assert p.ndim == 2 and p.shape == (len(y_true),2), p.shape
    p = np.clip(p, 1e-8, 1.0); p = p / p.sum(axis=1,keepdims=True)
    return {'accuracy':accuracy_score(y_true,y_pred),
            'macro_f1':f1_score(y_true,y_pred,average='macro',zero_division=0),
            'roc_auc':roc_auc_score(y_true,p[:,1]),
            'log_loss':log_loss(y_true,p,labels=[0,1])}

def record(status, seed, n, fit_sec=None, pred_sec=None, scores=None, err='', version=''):
    out = {'model':MODEL_ID,'dataset':'breast_cancer_wisconsin', 'seed':seed,
           'train_size':n,'status':status,'accuracy':np.nan,'macro_f1':np.nan,
           'roc_auc':np.nan,'log_loss':np.nan,'fit_seconds':fit_sec,
           'predict_seconds':pred_sec,'package_version':version,
           'error':str(err)[:800]}
    if scores: out.update(scores)
    return out

def save_results(rows):
    file=RESULTS / (MODEL_ID.lower().replace(' ','_')+'.csv')
    pd.DataFrame(rows).to_csv(file,index=False)
    print('Đã ghi:',file)
    display(pd.DataFrame(rows))
    return file

print('DATASET:', DATASET, '| shape:',X.shape,'| classes:',np.bincount(y))
print('CPU:',platform.processor() or platform.machine(), '| threads:',N_CPU_THREADS)
print('Mỗi seed dùng cùng split cố định; không dùng test để chọn hyperparameter.')

DATASET: /Users/thanhpc02/Downloads/Tabular_4SOTA_CPU/datasets/breast_cancer_wisconsin.csv | shape: (569, 30) | classes: [212 357]
CPU: arm | threads: 4
Mỗi seed dùng cùng split cố định; không dùng test để chọn hyperparameter.


## 3. Chạy mô hình gốc trên CPU và xuất kết quả

In [3]:
from tabdpt import TabDPTClassifier
import inspect, importlib.metadata, torch
rows=[]
for seed in SEEDS:
    for n in TRAIN_SIZES:
        xtr,ytr,xval,yval,xte,yte,ids,split=make_fold(seed,n)
        try:
            sig=inspect.signature(TabDPTClassifier)
            init_kwargs={}
            if 'device' in sig.parameters: init_kwargs['device']='cpu'
            if 'compile' in sig.parameters: init_kwargs['compile']=False
            if 'use_flash_attention' in sig.parameters: init_kwargs['use_flash_attention']=False
            model=TabDPTClassifier(**init_kwargs)
            t=time.perf_counter();model.fit(xtr,ytr);fit_sec=time.perf_counter()-t
            pred_sig=inspect.signature(model.predict_proba)
            opts={'context_size':min(n,256),'n_ensembles':1,'seed':seed,
                  'use_retrieval':False,'temperature':1.0,'permute_classes':False}
            opts={k:v for k,v in opts.items() if k in pred_sig.parameters}
            t=time.perf_counter();prob=model.predict_proba(xte,**opts);pred_sec=time.perf_counter()-t
            pred=model.classes_[np.argmax(prob,axis=1)] if hasattr(model,'classes_') else np.argmax(prob,axis=1)
            rows.append(record('ok',seed,n,fit_sec,pred_sec,metrics(yte,pred,prob),version=importlib.metadata.version('tabdpt')))
        except Exception as e:
            rows.append(record('failed',seed,n,err=repr(e),version=importlib.metadata.version('tabdpt')))
            print(f'FAILED seed={seed}, n={n}:',repr(e))
file=save_results(rows)

/Users/thanhpc02/Downloads/Tabular_4SOTA_CPU/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/thanhpc02/Downloads/Tabular_4SOTA_CPU/.venv/lib/python3.14/site-packages/tabdpt/estimator.py:85: UserWarning: Using TabDPT on CPU, so setting inf_batch_size to 16. This helps to prevent OOMs but can result in excessively slow inference, especially if retrieval is not being used. Set inf_batch_size explicitly to suppress this warning.
  warnings.warn(
/Users/thanhpc02/Downloads/Tabular_4SOTA_CPU/.venv/lib/python3.14/site-packages/tabdpt/estimator.py:85: UserWarning: Using TabDPT on CPU, so setting inf_batch_size to 16. This helps to prevent OOMs but can result in excessively slow inference, especially if retrieval is not being used. Set inf_batch_size explicitly to suppress this warning.
  warnings.w

Đã ghi: /Users/thanhpc02/Downloads/Tabular_4SOTA_CPU/results/tabdpt.csv


,model,dataset,seed,train_size,status,accuracy,macro_f1,roc_auc,log_loss,fit_seconds,predict_seconds,package_version,error
0,TabDPT,breast_cancer_wisconsin,42,50,ok,0.964912,0.961911,0.995701,0.101996,0.000722,0.153051,1.1.14,
1,TabDPT,breast_cancer_wisconsin,42,100,ok,0.964912,0.961911,0.996362,0.091542,0.000753,0.198533,1.1.14,


## 4. Ghi chú và giới hạn

TabDPT thay đổi API giữa các phiên bản. Cố định `tabdpt==1.1.*`; nếu CPU không được hỗ trợ bởi checkpoint/thư viện tương ứng, kết quả ghi `failed`. Không tự thay bằng XGBoost hoặc kNN. Tham số `context_size`/ensemble được giảm để thử CPU; muốn reproduce paper phải dùng cấu hình gốc.